<a href="https://colab.research.google.com/github/slomi23/ML_final/blob/main/model_experiment_XGBoost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone "https://github.com/slomi23/ML_fx.git"
!cd ML_fx/

fatal: destination path 'ML_fx' already exists and is not an empty directory.


# Load Data

In [2]:
import pandas as pd
import numpy as np
import os
import zipfile
import io

PROCCESSED_DATA_DIR = "./ML_fx/data/processed/"
train=pd.read_csv(os.path.join(PROCCESSED_DATA_DIR, "train_prepared.csv"))
print(train.head())


   Store  Dept        Date  Weekly_Sales  IsHoliday  Temperature  Fuel_Price  \
0      1     1  2010-02-05      24924.50          0        42.31       2.572   
1      1     1  2010-02-12      46039.49          1        38.51       2.548   
2      1     1  2010-02-19      41595.55          0        39.93       2.514   
3      1     1  2010-02-26      19403.54          0        46.63       2.561   
4      1     1  2010-03-05      21827.90          0        46.50       2.625   

   MarkDown1  MarkDown2  MarkDown3  ...  Type    Size  sales_lag_52  Year  \
0    5347.45      192.0       24.6  ...    20  151315       7998.55  2010   
1    5347.45      192.0       24.6  ...    20  151315       7998.55  2010   
2    5347.45      192.0       24.6  ...    20  151315       7998.55  2010   
3    5347.45      192.0       24.6  ...    20  151315       7998.55  2010   
4    5347.45      192.0       24.6  ...    20  151315       7998.55  2010   

   month_sin     month_cos   dow_sin   dow_cos  week_sin

# Split into train and val
val is little too big but it's because we wanted to fit in christmas (also fool year)

In [3]:
split_date = '2011-12-01'

# Create Train and Validation Sets
val_set = train[train['Date'] >= split_date]
train_set = train[train['Date'] < split_date]

y_train = train_set['Weekly_Sales']
X_train = train_set.drop(columns=['Weekly_Sales', 'Date'])
y_val = val_set['Weekly_Sales']
X_val = val_set.drop(columns=['Weekly_Sales', 'Date'])


print(f"Final Training Set Shape: {train_set.shape}")
print(f"Validation Set Shape: {val_set.shape}")
print(f"Validation Period: {val_set['Date'].min()} to {val_set['Date'].max()}")

Final Training Set Shape: (279085, 24)
Validation Set Shape: (142485, 24)
Validation Period: 2011-12-02 to 2012-10-26


In [4]:
!pip install wandb -q
!pip install xgboost==1.7.6

import wandb
import os

# Retrieve the secret from Kaggle Secrets

api_key = "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"
if api_key:
    wandb.login(key=api_key)
else:
    print("Warning: could not log in wandb ")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: slomi23 (slomi23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [16]:

import xgboost as xgb
import pandas as pd
import numpy as np
import wandb
import os
import joblib
from sklearn.metrics import mean_absolute_error

# Initialize W&B run
run = wandb.init(
    project="ML_fx_XGBoost_Walmart",
    name="XGBoost_run2",
    config={
        'objective': 'reg:squarederror',
        'eval_metric': 'mae',
        'learning_rate': 0.15,
        'max_depth': 6,
        'n_estimators': 500,
        'early_stopping_rounds': 40,
        'seed': 42,
        'verbosity': 1
    }
)

params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'mae',
    'learning_rate': 0.15,
    'max_depth': 6,
    'n_estimators': 500,
    'early_stopping_rounds': 40,
    'seed': 42,
    'verbosity': 1
}

model_params = {k: v for k, v in params.items() if k != 'early_stopping_rounds'}
model = xgb.XGBRegressor(**model_params)

# ---------------------------------------------------------
# NEW STYLE CALLBACK FOR XGBOOST >= 1.6
# ---------------------------------------------------------
class WandBCallback(xgb.callback.TrainingCallback):
    def after_iteration(self, model, epoch, evals_log):
        # evals_log structure: {'train': {'mae': [val]}, 'validation_0': {'mae': [val]}}
        if 'train' in evals_log and 'mae' in evals_log['train']:
            train_mae = evals_log['train']['mae'][-1]
        else:
            train_mae = None

        if 'validation_0' in evals_log and 'mae' in evals_log['validation_0']:
            val_mae = evals_log['validation_0']['mae'][-1]
        else:
            val_mae = None

        if train_mae is not None and val_mae is not None:
            wandb.log({
                "train_mae": train_mae,
                "val_mae": val_mae
            })
        return False # Return False to continue training

try:
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
        early_stopping_rounds=params['early_stopping_rounds'],
        callbacks=[WandBCallback()] # Instantiate the new callback class
    )

    print(f"Best Iteration: {model.best_iteration}")
    run.summary["best_val_mae"] = model.evals_result()["validation_0"]["mae"][model.best_iteration]

except Exception as e:
    print(f"Error during fitting: {e}")
    raise e
# ---------------------------------------------------------
# 4. FINAL EVALUATION & LOGGING
# ---------------------------------------------------------
y_pred = model.predict(X_val)
y_train_pred = model.predict(X_train)

is_holiday = X_val['IsHoliday'].values
is_holiday_train = X_train['IsHoliday'].values
weights = np.where(is_holiday, 5, 1)
weights_train = np.where(is_holiday_train, 5, 1)

final_wmae = np.average(np.abs(y_val - y_pred), weights=weights)
final_mae = mean_absolute_error(y_val, y_pred)

train_wmae = np.average(np.abs(y_train - y_train_pred), weights=weights_train)
train_mae = mean_absolute_error(y_train, y_train_pred)

if run:
    wandb.log({"final_validation_wmae": final_wmae, "final_validation_mae": final_mae})
    wandb.log({"train_wmae": train_wmae, "train_mae": train_mae})

    importance = model.feature_importances_
    feature_names = X_train.columns.tolist()
    importance_df = pd.DataFrame({'feature': feature_names, 'importance': importance})
    importance_df = importance_df.sort_values(by='importance', ascending=False)

    table = wandb.Table(dataframe=importance_df.head(20))
    wandb.log({"feature_importance_table": table})

    artifact = wandb.Artifact("xgboost-model-1", type="model")
    joblib.dump(model, "model.joblib")
    artifact.add_file("model.joblib")
    run.log_artifact(artifact)

print(f"Final Train WMAE: {train_wmae:.2f}")
print(f"Final Train MAE: {train_mae:.2f}")
print(f"Final Validation WMAE: {final_wmae:.2f}")
print(f"Final Validation MAE: {final_mae:.2f}")

if run:
    wandb.finish()


/usr/local/lib/python3.12/dist-packages/xgboost/sklearn.py:835: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/xgboost/sklearn.py:835: UserWarning: `callbacks` in `fit` method is deprecated for better compatibility with scikit-learn, use `callbacks` in constructor or`set_params` instead.
  warnings.warn(


Best Iteration: 223
Final Train WMAE: 2750.41
Final Train MAE: 2627.82
Final Validation WMAE: 2588.36
Final Validation MAE: 2428.29


final_validation_mae,▁
final_validation_wmae,▁
train_mae,▁
train_wmae,▁
best_val_mae,2428.2857
final_validation_mae,2428.2857
final_validation_wmae,2588.3573
train_mae,2627.82488
train_wmae,2750.40953
